# Практика · Що змінює масштаб

> Теорія: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

Зошит самодостатній: усе, що тут відбувається, пояснено на місці, лекцію
відкривати не обовʼязково.

**Задача, яку ми розвʼязуємо.** Зробити те, що роблять лабораторії перед тим,
як витратити мільйони на навчання великої моделі: **побудувати криву масштабу**.
Тобто заміряти, як якість моделі залежить від того, скільки в неї ваг і скільки
їй дали даних, — і чесно сказати, що з цієї кривої можна передбачити, а чого не
можна.

Що зробимо:

1. зберемо корпус українських речень **із цієї машини** — це переклади
   інтерфейсів програм, які в тебе встановлені;
2. **порахуємо ваги** шести моделей і побачимо, що «розмір моделі» — це не одне
   число, а щонайменше два, і вони дають різні відповіді;
3. побудуємо криву **якість від обсягу даних** на 483-кратному діапазоні —
   безкоштовно, лічильниками;
4. напишемо **підгін прямої на логарифмічній сітці** з нуля й звіримо з `numpy`;
5. перевіримо, **скільки обіцяє екстраполяція** — продовжимо пряму й порівняємо
   обіцянку із заміром у тій самій точці;
6. навчимо **чотири трансформери різного розміру** й дістанемо показник
   степеневого закону — двічі, на двох різних осях;
7. на **одному й тому самому виході однієї й тієї самої моделі** порахуємо сімʼю
   метрик від гладкої до розривної й побачимо, звідки береться «стрибок здібності».

> ⏱ Зошит навчає чотири невеликі мовні моделі. Заміряно: **близько чотирьох
> хвилин процесорного часу** (241 с) на машині з чотирма ядрами й без
> відеокарти, з яких на саме навчання йде близько трьох хвилин. За настінним
> годинником на завантаженій машині вийде більше — у нас на тому ж прогоні
> 270 с. Саме тому зошит скрізь друкує процесорний час, а не годинник: він
> єдиний, що не залежить від того, чим ще зайнята машина.

In [ ]:
# Кількість потоків фіксуємо ДО імпорту numpy і torch. Без цього бібліотеки
# лінійної алгебри розповзаються по всіх ядрах, потоки крутяться в очікуванні,
# і це очікування рахується як робота: замір часу спотворюється в десятки разів.
import os
for var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[var] = '1'

import re, glob, math, time, gettext, random, collections, statistics
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.set_num_threads(1)

SEED = 0                      # одне зерно на весь зошит: усе відтворюється
started_at = time.process_time()

print('numpy', np.__version__, '· torch', torch.__version__)
print('потоків у torch:', torch.get_num_threads())

## 1 · Корпус: українські речення з цієї машини

Курс не тримає текстів у репозиторії. Замість цього ми беремо те, що вже лежить
у кожній системі з українською локаллю: **скомпільовані файли перекладів**
`/usr/share/locale/uk/LC_MESSAGES/*.mo`. У кожному з них — пари «англійський
рядок інтерфейсу → український переклад», і писали ці переклади живі люди.

Формат `.mo` двійковий, але читати його вручну не треба: стандартний модуль
`gettext` уміє це сам.

⚠️ **Твої числа не збіжаться з тими, що надруковані нижче в цьому файлі.**
Набір встановлених програм у кожного свій, тож і корпус свій. Що відтворюється —
це **форма** залежностей: нахили кривих і співвідношення між ними. Абсолютні
підрахунки — ні. Тому зошит скрізь друкує **свої** числа, а висновки формулює
через нахили.

In [ ]:
# Токенізуємо тільки українські слова: латиниця в цьому корпусі — це імена
# файлів, ключі конфігів і назви програм, тобто не мова, а сміття для нашої задачі.
UKRAINIAN_WORD = re.compile(r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*")

def ukrainian_words(text):
    return UKRAINIAN_WORD.findall(text.lower())

def load_translations(min_chars=20):
    """Усі українські переклади довші за min_chars символів."""
    sentences = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)._catalog
        except Exception:
            continue                     # пошкоджений або нестандартний файл — пропускаємо
        for source, translated in catalog.items():
            if not isinstance(source, str) or not isinstance(translated, str):
                continue
            # службовий заголовок кожного каталогу — не речення
            if len(translated) > min_chars and 'Project-Id' not in translated:
                sentences.append(translated)
    return sentences

raw_sentences = load_translations()
print('знайдено перекладів:', len(raw_sentences))
print('приклад:', raw_sentences[len(raw_sentences) // 3][:120])

### Фільтр довжини й поділ на три частини

Беремо речення від 4 до 30 слів. Коротші не дають контексту, довші коштують
памʼяті й трапляються рідко.

Ділимо на три частини, і кожна має свою роль:

* **навчальна** — на ній моделі вчаться;
* **відкладена** — на ній добирають налаштування, якщо доводиться добирати;
* **перевірна** — на ній міряють остаточне число, і більше ні на що вона не йде.

Перед поділом обовʼязково **перемішуємо**. Файли читаються в абетковому порядку
програм, тож «перші 80 %» — це не менший корпус, а вужчий: пів сотні програм
замість двохсот. Крива, побудована на такому поділі, показала б не залежність
від обсягу, а залежність від різноманіття тем.

In [ ]:
MAX_LEN = 32                      # найдовша послідовність із <bos> і <eos>
PAD, BOS, EOS, UNK = 0, 1, 2, 3   # службові номери

tokenized = [ukrainian_words(s) for s in raw_sentences]
tokenized = [w for w in tokenized if 4 <= len(w) <= MAX_LEN - 2]

random.Random(SEED).shuffle(tokenized)      # перемішуємо ДО поділу
n_total = len(tokenized)
cut_train, cut_hold = int(0.8 * n_total), int(0.9 * n_total)
train_words = tokenized[:cut_train]
hold_words  = tokenized[cut_train:cut_hold]
test_words  = tokenized[cut_hold:]

# Словник будуємо ТІЛЬКИ по навчальній частині: інакше модель побічно
# підглядає в перевірну ще до першого кроку навчання.
word_counts = collections.Counter(w for s in train_words for w in s)
vocabulary = ['<pad>', '<bos>', '<eos>', '<unk>'] + \
             [w for w, c in word_counts.most_common() if c >= 5]
word_to_id = {w: i for i, w in enumerate(vocabulary)}
VOCAB = len(vocabulary)

def encode(words):
    return [BOS] + [word_to_id.get(w, UNK) for w in words] + [EOS]

train_ids = [encode(s) for s in train_words]
hold_ids  = [encode(s) for s in hold_words]
test_ids  = [encode(s) for s in test_words]
train_tokens = sum(len(s) - 1 for s in train_ids)   # скільки разів модель щось передбачає

print(f'речень: навчальна {len(train_ids)} · відкладена {len(hold_ids)} · '
      f'перевірна {len(test_ids)}')
print(f'словник {VOCAB} слів · токенів у навчальній частині {train_tokens}')

## 2 · «Розмір моделі» — це не одне число

Тут починається перший урок теми, і він коштує нуль секунд обчислень.

Наша модель — маленький трансформер. У ньому три групи ваг:

* **таблиця ембедингів** — по рядку на кожне слово словника: `VOCAB × d`;
* **тіло** — шари уваги й повнозвʼязні шари, які, власне, й обробляють текст;
* **вихідна проєкція** — назад зі `d` чисел у `VOCAB` оцінок, ще `VOCAB × d`.

Дві з трьох груп ростуть **лінійно** зі шириною `d` і пропорційні розміру
словника. Тіло росте як **`d²`**. Тому «збільшити модель удвічі» означає різні
речі залежно від того, що ти рахуєш, — і зараз ми це побачимо числом.

In [ ]:
class TinyLM(nn.Module):
    """Маленька мовна модель: ембединги + шари трансформера + вихідна проєкція."""

    def __init__(self, vocab, d, layers=2, heads=4):
        super().__init__()
        self.token_emb = nn.Embedding(vocab, d, padding_idx=PAD)
        self.pos_emb = nn.Embedding(MAX_LEN, d)
        layer = nn.TransformerEncoderLayer(
            d, heads, dim_feedforward=4 * d, batch_first=True,
            dropout=0.0, norm_first=True)
        self.body = nn.TransformerEncoder(layer, layers)
        self.head = nn.Linear(d, vocab)

    def forward(self, x):
        length = x.size(1)
        h = self.token_emb(x) + self.pos_emb(torch.arange(length))
        # маска майбутнього: передбачаючи слово, модель не має права його бачити
        future = torch.triu(torch.full((length, length), float('-inf')), 1)
        return self.head(self.body(h, mask=future, src_key_padding_mask=(x == PAD)))


def count_parts(model):
    """Скільки ваг у кожній групі."""
    body = sum(p.numel() for p in model.body.parameters())
    emb = model.token_emb.weight.numel() + model.pos_emb.weight.numel()
    head = sum(p.numel() for p in model.head.parameters())
    return dict(body=body, emb=emb, head=head, total=body + emb + head)


WIDTHS = [24, 32, 48, 64]
parts_table = []
for width in WIDTHS:
    parts = count_parts(TinyLM(VOCAB, width))
    parts_table.append((width, parts))
    print(f'd={width:3} · усього {parts["total"]:8} · тіло {parts["body"]:7} '
          f'({100 * parts["body"] / parts["total"]:5.2f} %) · '
          f'ембединги {parts["emb"]:7} · вихід {parts["head"]:7}')

range_total = parts_table[-1][1]['total'] / parts_table[0][1]['total']
range_body = parts_table[-1][1]['body'] / parts_table[0][1]['body']
print(f'\nвід найменшої до найбільшої: усіх ваг більше в {range_total:.2f} раза, '
      f'ваг тіла — в {range_body:.2f} раза')

Різниця не косметична. Якщо крайні моделі відрізняються за всіма вагами в
кілька разів, а за вагами тіла — у кілька десятків разів, то **той самий замір
дасть два різні показники степеневого закону**, і відрізнятимуться вони не в
четвертому знаку. Ми повернемось до цього після навчання.

Причина видна з арифметики: таблиця ембедингів і вихідна проєкція разом дають
`2 · VOCAB · d` ваг, а тіло — приблизно `12 · шарів · d²`. При нашому словнику й
малих `d` перший доданок у рази більший.

In [ ]:
# Перевіримо арифметику: формула має збігтися з тим, що порахував torch.
for width, parts in parts_table:
    formula_body = 2 * (12 * width * width + 13 * width)   # 2 шари
    formula_emb = VOCAB * width + MAX_LEN * width
    formula_head = VOCAB * width + VOCAB
    assert formula_body == parts['body'], 'формула тіла розійшлася'
    assert formula_emb == parts['emb'], 'формула ембедингів розійшлася'
    assert formula_head == parts['head'], 'формула виходу розійшлася'
print('✅ формули збігаються з підрахунком torch для всіх', len(parts_table), 'розмірів')
print('частка тіла при найменшій ширині:',
      f'{100 * parts_table[0][1]["body"] / parts_table[0][1]["total"]:.2f} %')

## 3 · Крива за обсягом даних — безкоштовно

Щоб побачити степеневий закон, нейромережа не потрібна. Достатньо найпростішої
моделі мови — **біграми**: імовірність наступного слова оцінюємо тим, скільки
разів воно траплялося після поточного.

Дві деталі, без яких біграма не працює:

* **згладжування**: пари, якої не було в навчанні, модель має оцінити не нулем,
  інакше правдоподібність усього речення стане нулем, а логарифм — мінус
  нескінченністю. Додаємо до кожного лічильника невелику сталу;
* **інтерполяція з уніграмою**: там, де про пару нічого не відомо, спираємось на
  просту частоту слова.

Міряємо в **натах на токен** — це середній `−log p` правильного слова.
Нуль означав би ідеальне передбачення; `log(VOCAB)` — повне незнання.

In [ ]:
def bigram_nats(train_rows, test_rows, vocab, lam=0.7, add=0.1):
    """Середній -log p правильного токена. Менше — краще."""
    unigram = collections.Counter()
    bigram = collections.Counter()
    context = collections.Counter()
    for row in train_rows:
        for prev, nxt in zip(row, row[1:]):
            bigram[(prev, nxt)] += 1
            context[prev] += 1
            unigram[nxt] += 1
    unigram_total = sum(unigram.values())

    total_logp, n_predictions = 0.0, 0
    for row in test_rows:
        for prev, nxt in zip(row, row[1:]):
            p_uni = (unigram[nxt] + add) / (unigram_total + add * vocab)
            # якщо контексту не бачили жодного разу, спираємось лише на уніграму
            p_bi = (bigram[(prev, nxt)] + add) / (context[prev] + add * vocab) \
                if context[prev] else p_uni
            total_logp -= math.log(lam * p_bi + (1 - lam) * p_uni)
            n_predictions += 1
    return total_logp / n_predictions


sampler = random.Random(SEED)
data_sizes, data_nats = [], []
print('токенів у навчанні → нат на токен')
for fraction in (0.002, 0.005, 0.01, 0.03, 0.1, 0.3, 1.0):
    # ВИПАДКОВА частка, а не перші N рядків: корпус лежить у порядку програм,
    # тож префікс дав би вужчу тему, а не менший обсяг.
    subset = sampler.sample(train_ids, max(2, int(fraction * len(train_ids))))
    tokens = sum(len(s) - 1 for s in subset)
    value = bigram_nats(subset, test_ids, VOCAB)
    data_sizes.append(tokens)
    data_nats.append(value)
    print(f'  {tokens:8} → {value:.4f}')
print(f'\nдіапазон обсягу: {data_sizes[-1] / data_sizes[0]:.0f}-кратний')

## 4 · Пряма на логарифмічній сітці, написана з нуля

Степеневий закон — це залежність виду `y = a · x^b`. Прологарифмувавши обидві
частини, дістаємо `log y = log a + b · log x`, тобто **пряму**. Тому показник `b`
знаходять звичайною прямою по точках `(log x, log y)`.

Напишемо цей підгін самі — це вісім рядків шкільної формули, — а потім звіримо
з `numpy.polyfit`. Мета звірки не в тому, щоб недовіряти `numpy`, а в тому, щоб
побачити, що всередині немає магії.

`R²` — частка розкиду, яку пояснила пряма. Одиниця означає, що точки лежать на
прямій ідеально; нуль — що пряма не краща за горизонтальну лінію на середньому
рівні.

In [ ]:
def fit_power_law(xs, ys):
    """Пряма по точках (log x, log y). Повертає (показник, log a, R²)."""
    log_x = [math.log(v) for v in xs]
    log_y = [math.log(v) for v in ys]
    n = len(log_x)
    mean_x = sum(log_x) / n
    mean_y = sum(log_y) / n
    covariance = sum((a - mean_x) * (b - mean_y) for a, b in zip(log_x, log_y))
    variance = sum((a - mean_x) ** 2 for a in log_x)
    slope = covariance / variance
    intercept = mean_y - slope * mean_x
    predicted = [intercept + slope * a for a in log_x]
    ss_res = sum((b - p) ** 2 for b, p in zip(log_y, predicted))
    ss_tot = sum((b - mean_y) ** 2 for b in log_y)
    return slope, intercept, 1 - ss_res / ss_tot


def power_law_value(slope, intercept, x):
    """Що обіцяє підігнана пряма в точці x."""
    return math.exp(intercept + slope * math.log(x))


data_slope, data_intercept, data_r2 = fit_power_law(data_sizes, data_nats)
numpy_slope, numpy_intercept = np.polyfit(np.log(data_sizes), np.log(data_nats), 1)

assert np.allclose([data_slope, data_intercept], [numpy_slope, numpy_intercept]), \
    'наш підгін розійшовся з numpy!'
print('✅ збігається з numpy.polyfit')
print(f'показник за обсягом даних: {data_slope:.4f} · R² {data_r2:.4f}')
print(f'тобто вдесятеро більше даних → втрата множиться на '
      f'{10 ** data_slope:.4f}, тобто падає на {100 * (1 - 10 ** data_slope):.2f} %')

### Малюємо криву в двох системах координат

Одні й ті самі сім точок. Ліворуч — звичайні осі, праворуч — логарифмічні.
Праворуч видно пряму; ліворуч видно тільки те, що «спочатку швидко, потім
повільно». Це не два різні явища, а одне явище й дві лінійки.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].plot(data_sizes, data_nats, 'o-', color='#c2185b')
axes[0].set_xlabel('токенів у навчанні')
axes[0].set_ylabel('нат на токен')
axes[0].set_title('звичайні осі')

axes[1].plot(data_sizes, data_nats, 'o', color='#c2185b', label='замір')
grid_x = np.geomspace(min(data_sizes), max(data_sizes), 50)
axes[1].plot(grid_x, [power_law_value(data_slope, data_intercept, v) for v in grid_x],
             '--', color='#0f766e', label=f'пряма, показник {data_slope:.4f}')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('токенів у навчанні (лог)')
axes[1].set_ylabel('нат на токен (лог)')
axes[1].set_title('логарифмічні осі')
axes[1].legend(fontsize=9)

figure.tight_layout()
plt.show()
print('R² прямої на логарифмічній сітці:', round(data_r2, 4))

## 5 · Скільки обіцяє екстраполяція

Тепер найважливіша перевірка зошита. Уявімо, що ми заміряли лише **чотири
найменші** точки — так і буває, коли великий прогін ще попереду. Продовжимо
пряму до найбільшого обсягу й порівняємо обіцянку з тим, що ми насправді
заміряли в тій самій точці.

Це не вправа на акуратність. Це головний спосіб, у який криві масштабу вводять
в оману: `R²` високий, точки лягли красиво, і здається, що продовження прямої —
теж замір. Воно ним не є.

In [ ]:
n_short = 4
short_slope, short_intercept, short_r2 = fit_power_law(
    data_sizes[:n_short], data_nats[:n_short])

target_x = data_sizes[-1]
promised = power_law_value(short_slope, short_intercept, target_x)
measured = data_nats[-1]

print(f'підгін по перших {n_short} точках: показник {short_slope:.4f} · R² {short_r2:.4f}')
print(f'підгін по всіх {len(data_sizes)}    : показник {data_slope:.4f} · R² {data_r2:.4f}')
print()
print(f'у точці {target_x} токенів')
print(f'  коротка пряма обіцяє : {promised:.4f} ната')
print(f'  заміряно насправді   : {measured:.4f} ната')
print(f'  розрив               : {measured - promised:+.4f} ната')
print(f'  тобто пряма помилилась на {abs(measured - promised) / measured * 100:.1f} % '
      f'значення, екстраполюючи всього в {target_x / data_sizes[n_short - 1]:.0f} разів')

Показник, порахований по чотирьох точках, і показник по всіх семи — **різні
числа**. Крива не є ідеальною прямою: вона повільно згинається, і напрямок згину
з чотирьох точок не видно.

І друга обіцянка, яку пряма дає мовчки. У формулі `y = a · x^b` при `b < 0`
значення прямує до нуля, коли `x` росте. Тобто **чистий степеневий закон обіцяє,
що з достатньою кількістю даних втрата стане нульовою**. Це неможливо: у мові є
незнищенна невизначеність — після «я поїхав до» може стояти будь-яке місто.
Порахуємо, чого коштувала б навіть скромна ціль.

In [ ]:
def tokens_needed(target_nats):
    """Скільки токенів обіцяє пряма для заданого рівня втрати."""
    return math.exp((math.log(target_nats) - data_intercept) / data_slope)

print(f'заміряно зараз: {data_nats[-1]:.4f} ната на {data_sizes[-1]} токенах\n')
for target in (5.0, 4.5, 4.0, 3.0):
    need = tokens_needed(target)
    print(f'  щоб дійти {target:.1f} ната, пряма вимагає {need:.3e} токенів '
          f'— це в {need / data_sizes[-1]:.0f} разів більше, ніж у нас є')
print('\nЖодне з цих чисел не заміряне. Це те, що обіцяє пряма, '
      'а обіцянка прямої — гіпотеза.')

## 6 · Крива за розміром моделі

Тепер навчимо чотири трансформери різної ширини. Усе інше **тримаємо сталим**:
ті самі дані, та сама кількість кроків, той самий крок навчання, те саме зерно.
Міняється лише `d`.

Одразу назвемо, чим ця постановка чесна, а чим ні:

* **чесно**, що дані й кількість кроків однакові: ми міряємо саме розмір;
* **нечесно** те, що крок навчання один на всі розміри. Для великих моделей
  зазвичай потрібен менший крок, тож найширшій моделі ми, найімовірніше,
  трохи заважаємо. Це означає, що заміряний показник — **нижня оцінка**;
* **нечесно** називати цей бюджет сталим. Кількість кроків стала, а обчислень
  ширша модель зʼїдає більше — приблизно пропорційно кількості ваг.

In [ ]:
def pad_batch(rows):
    longest = max(len(r) for r in rows)
    return torch.tensor([r + [PAD] * (longest - len(r)) for r in rows])


def train_model(model, rows, steps, lr, seed, batch=64):
    """Навчання мовної моделі. Повертає витрачений процесорний час."""
    torch.manual_seed(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    schedule = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=steps, pct_start=0.1)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
    picker = random.Random(seed)
    indices = list(range(len(rows)))
    t0 = time.process_time()
    for _ in range(steps):
        batch_rows = pad_batch([rows[i] for i in picker.sample(indices, batch)])
        logits = model(batch_rows[:, :-1])
        loss = loss_fn(logits.reshape(-1, VOCAB), batch_rows[:, 1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        # обрізаємо довжину градієнта: рідкісний величезний крок інакше
        # викидає ваги в область, з якої модель не повертається
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        schedule.step()
    return time.process_time() - t0


@torch.no_grad()
def evaluate_nats(model, rows, batch=128):
    """Середній -log p правильного токена на перевірній частині."""
    model.eval()
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD, reduction='sum')
    total, n = 0.0, 0
    for i in range(0, len(rows), batch):
        x = pad_batch(rows[i:i + batch])
        total += loss_fn(model(x[:, :-1]).reshape(-1, VOCAB),
                         x[:, 1:].reshape(-1)).item()
        n += (x[:, 1:] != PAD).sum().item()
    model.train()
    return total / n


STEPS, LR = 250, 1e-2
trained = {}
size_all, size_body, size_nats = [], [], []
print(f'{STEPS} кроків, крок навчання {LR}, зерно {SEED}\n')
for width in WIDTHS:
    torch.manual_seed(SEED)
    model = TinyLM(VOCAB, width)
    parts = count_parts(model)
    spent = train_model(model, train_ids, STEPS, LR, SEED)
    value = evaluate_nats(model, test_ids)
    trained[width] = model
    size_all.append(parts['total'])
    size_body.append(parts['body'])
    size_nats.append(value)
    print(f'd={width:3} · усього ваг {parts["total"]:8} · тіло {parts["body"]:7} '
          f'→ {value:.4f} ната · {spent:.0f} с процесорних')

### Два показники з одного заміру

Ті самі чотири точки. Один раз кладемо на горизонтальну вісь **усі ваги**,
другий раз — **ваги тіла**. Втрата в обох випадках та сама.

In [ ]:
slope_all, intercept_all, r2_all = fit_power_law(size_all, size_nats)
slope_body, intercept_body, r2_body = fit_power_law(size_body, size_nats)

print(f'вісь «усі ваги»  : показник {slope_all:.4f} · R² {r2_all:.4f}')
print(f'вісь «ваги тіла» : показник {slope_body:.4f} · R² {r2_body:.4f}')
print(f'відношення показників: {slope_all / slope_body:.2f}')
print()
print('Це не помилка й не шум. Обидва числа правильні — вони відповідають на')
print('різні питання, бо на двох осях однаковій зміні втрати відповідає різна')
print('зміна «розміру». Показник степеневого закону без назви осі не є числом.')

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, xs, slope, intercept, name in (
        (axes[0], size_all, slope_all, intercept_all, 'усі ваги'),
        (axes[1], size_body, slope_body, intercept_body, 'ваги тіла')):
    ax.plot(xs, size_nats, 'o', color='#c2185b', label='замір')
    grid_x = np.geomspace(min(xs), max(xs), 50)
    ax.plot(grid_x, [power_law_value(slope, intercept, v) for v in grid_x],
            '--', color='#0f766e', label=f'показник {slope:.4f}')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(name + ' (лог)')
    ax.legend(fontsize=9)
axes[0].set_ylabel('нат на токен (лог)')
figure.suptitle('одні й ті самі чотири моделі, дві різні осі', fontsize=11)
figure.tight_layout()
plt.show()

## 7 · Три метрики на одному й тому самому виході

Тепер про «здібності, що зʼявляються стрибком».

Візьмемо 150 речень із перевірної частини. Даємо моделі перші три слова —
і просимо дописати решту, щоразу беручи **найімовірніше** наступне слово.
Вихід моделі ми більше не чіпаємо: усі метрики нижче рахуються **на тих самих
рядках**. Міняється тільки лінійка.

* **точний збіг** — вихід збігся з еталоном цілком, слово в слово. Або 1, або 0;
* **F1 по токенах** — скільки спільних слів у виході й еталоні, без урахування
  порядку. Дає часткову оцінку;
* **точність по токенах** — на кожному місці окремо: чи вгадала модель наступне
  слово, якщо всі попередні їй підказали правильно. Найгладкіша з трьох.

In [ ]:
# Беремо речення, у яких після підказки лишається щонайменше три слова.
PROMPT_LEN = 4                    # <bos> і три слова
long_rows = [r for r in test_ids if len(r) >= PROMPT_LEN + 3][:150]
prompts = [r[:PROMPT_LEN] for r in long_rows]
references = [r[PROMPT_LEN:] for r in long_rows]

print('прикладів:', len(references))
print('довжина продовження: медіана',
      statistics.median(len(r) for r in references),
      '· середня', round(statistics.mean(len(r) for r in references), 2),
      '· максимум', max(len(r) for r in references))

reference_tokens = collections.Counter(t for r in references for t in r)
n_ref_tokens = sum(reference_tokens.values())
print(f'частка <eos> серед еталонних токенів: '
      f'{reference_tokens[EOS] / n_ref_tokens:.4f}')
print(f'частка <unk> серед еталонних токенів: '
      f'{reference_tokens[UNK] / n_ref_tokens:.4f}')

In [ ]:
@torch.no_grad()
def greedy_continuations(model):
    """Дописуємо кожну підказку рівно на довжину її еталона."""
    model.eval()
    grown = [list(p) for p in prompts]
    for step in range(max(len(r) for r in references)):
        x = pad_batch([s[-MAX_LEN:] for s in grown])
        logits = model(x)
        for i in range(len(grown)):
            if step < len(references[i]):
                grown[i].append(int(logits[i, len(grown[i]) - 1].argmax()))
    model.train()
    return [grown[i][PROMPT_LEN:] for i in range(len(grown))]


def token_f1(generated, reference):
    """Скільки спільних токенів, без урахування порядку."""
    a = collections.Counter(generated)
    b = collections.Counter(reference)
    overlap = sum((a & b).values())
    if overlap == 0:
        return 0.0
    return 2 * overlap / (sum(a.values()) + sum(b.values()))


@torch.no_grad()
def teacher_forced_accuracy(model):
    """Частка місць, де найімовірніше слово збіглося з правильним."""
    model.eval()
    rows = [list(prompts[i]) + list(references[i]) for i in range(len(prompts))]
    x = pad_batch([r[:MAX_LEN] for r in rows])
    predicted = model(x).argmax(-1)
    hit, total = 0, 0
    for i, reference in enumerate(references):
        for j, token in enumerate(reference):
            if PROMPT_LEN + j >= x.size(1):
                break
            hit += int(int(predicted[i, PROMPT_LEN + j - 1]) == token)
            total += 1
    model.train()
    return hit / total


print(f'{"d":>4} {"нат":>8} {"точний збіг":>12} {"F1":>8} {"точність по токенах":>21}')
metrics = {}
for width in WIDTHS:
    generated = greedy_continuations(trained[width])
    exact = statistics.mean(
        int(g == r) for g, r in zip(generated, references))
    f1 = statistics.mean(
        token_f1(g, r) for g, r in zip(generated, references))
    accuracy = teacher_forced_accuracy(trained[width])
    metrics[width] = dict(generated=generated, exact=exact, f1=f1, accuracy=accuracy)
    print(f'{width:>4} {size_nats[WIDTHS.index(width)]:>8.4f} {exact:>12.4f} '
          f'{f1:>8.4f} {accuracy:>21.4f}')

### Підлога метрики: скільки дає відповідь, яка нічого не знає

Число саме по собі не означає нічого, поки поруч немає **підлоги** — того, що
дає найтупіша можлива відповідь. Порахуємо три такі відповіді. Жодна з них не
дивиться ні на підказку, ні на модель.

In [ ]:
# Найчастіший ТОКЕН, а не найчастіше слово: службові <eos> і <unk> — теж токени,
# і саме <eos> найчастіший. Рахувати підлогу без них означало б занизити її.
token_counts = collections.Counter(t for row in train_ids for t in row[1:])
most_common_id = token_counts.most_common(1)[0][0]
most_common_token = vocabulary[most_common_id]

floors = {
    'усе <eos>': [[EOS] * len(r) for r in references],
    'усе <unk>': [[UNK] * len(r) for r in references],
    '<unk>, а в кінці <eos>': [[UNK] * (len(r) - 1) + [EOS] for r in references],
    f'усе {most_common_token!r}': [[most_common_id] * len(r) for r in references],
}
print('F1 тривіальних відповідей:')
best_floor = 0.0
for name, answer in floors.items():
    value = statistics.mean(token_f1(g, r) for g, r in zip(answer, references))
    best_floor = max(best_floor, value)
    print(f'  {name:<24} {value:.4f}')

print(f'\nнайвища підлога F1: {best_floor:.4f}')
print('F1 наших моделей:', ' · '.join(f'{metrics[w]["f1"]:.4f}' for w in WIDTHS))
above = [w for w in WIDTHS if metrics[w]['f1'] > best_floor]
print(f'моделей, що перевищили підлогу: {len(above)} з {len(WIDTHS)}')

# підлога точності по токенах: завжди називати найчастіше слово
floor_accuracy = sum(1 for r in references for t in r if t == most_common_id) \
    / n_ref_tokens
print(f'\nнайчастіший токен навчання: {most_common_token!r}')
print(f'підлога точності по токенах: {floor_accuracy:.4f}')
print('точність наших моделей:',
      ' · '.join(f'{metrics[w]["accuracy"]:.4f}' for w in WIDTHS))
print('запас над підлогою:',
      ' · '.join(f'{metrics[w]["accuracy"] - floor_accuracy:+.4f}' for w in WIDTHS))

Ось чому підлогу треба рахувати завжди. F1 по токенах виглядає як розумна
часткова оцінка — а насправді значну частину його значення дає те, що кожен
еталон закінчується службовим `<eos>`, і будь-яка відповідь потрібної довжини
має шанс у нього поцілити.

## 8 · Звідки береться «стрибок»

Тепер головна вправа теми. Візьмемо **той самий жадібний вихід тих самих
моделей** і порахуємо сімʼю метрик: чи збіглися перші `k` токенів, для `k` від 1
до 6. Це одна й та сама відповідь, оцінена шістьма лінійками різної строгості.

In [ ]:
K_MAX = 6

def prefix_match(generated_list, k):
    """Частка прикладів, де збіглися перші k токенів."""
    ok, total = 0, 0
    for generated, reference in zip(generated_list, references):
        if len(reference) < k:
            continue
        total += 1
        ok += int(generated[:k] == reference[:k])
    return ok / total if total else float('nan')


prefix_table = {}
header = ' '.join(f'k={k:<7}' for k in range(1, K_MAX + 1))
print(f'{"d":>4}  {header}')
for width in WIDTHS:
    row = [prefix_match(metrics[width]['generated'], k) for k in range(1, K_MAX + 1)]
    prefix_table[width] = row
    print(f'{width:>4}  ' + ' '.join(f'{v:<9.4f}' for v in row))

Подивись на таблицю по стовпчиках. Ліворуч (`k=1`) числа помітно різні —
метрика бачить різницю між моделями. Праворуч усі числа однакові й дорівнюють
нулю — метрика не бачить нічого.

Тепер найважливіше: **чому саме так**. Якщо модель вгадує кожен наступний токен
із імовірністю `p`, то ймовірність вгадати `k` поспіль приблизно `p^k`. Ця
формула перетворює **пряму** на **хокейну ключку**, не змінюючи в моделі нічого.
Порахуємо її від нашого ж заміру `k=1` і покладемо поруч.

In [ ]:
print('припущення: збіг k токенів ≈ (збіг одного)^k\n')
print(f'{"d":>4} {"p=k1":>8} ' + ' '.join(f'{"k=" + str(k):>18}' for k in range(2, 5)))
for width in WIDTHS:
    p = prefix_table[width][0]
    cells = []
    for k in range(2, 5):
        cells.append(f'{p ** k:.4f}/{prefix_table[width][k - 1]:.4f}')
    print(f'{width:>4} {p:>8.4f} ' + ' '.join(f'{c:>18}' for c in cells))
print('\n(у клітинці: передбачення p^k / заміряне значення)')

In [ ]:
# Малюємо те саме: гладка величина ліворуч, її ж степені праворуч.
figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].plot(size_all, [prefix_table[w][0] for w in WIDTHS], 'o-', color='#c2185b')
axes[0].axhline(floor_accuracy, ls=':', color='#5b6b7c', label='підлога')
axes[0].set_xscale('log')
axes[0].set_xlabel('усі ваги (лог)')
axes[0].set_ylabel('збіг першого токена')
axes[0].set_title('гладка метрика: k = 1')
axes[0].legend(fontsize=9)

# ліва межа — трохи нижче за найгіршу заміряну модель, права — одиниця:
# ймовірності, більшої за одиницю, не буває, а p^k при p > 1 росте вгору
smooth = np.linspace(min(prefix_table[w][0] for w in WIDTHS) * 0.5, 1.0, 60)
for k, color in zip((1, 2, 4, 8), ('#c2185b', '#c2620f', '#0f766e', '#5b6b7c')):
    axes[1].plot(smooth, smooth ** k, color=color, label=f'k = {k}')
axes[1].set_xlabel('імовірність вгадати один токен')
axes[1].set_ylabel('імовірність вгадати всі k')
axes[1].set_title('та сама величина, піднесена до степеня k')
axes[1].legend(fontsize=9)

figure.tight_layout()
plt.show()

Праворуч жодних даних немає — це чиста арифметика. І саме вона малює те, що в
статтях про великі моделі показують як «здібність, що виникла раптово»: поки
`p` малий, `p^k` невідрізненний від нуля; щойно `p` переходить певний рівень,
`p^k` злітає. Модель при цьому поводиться гладко.

Це не доводить, що всі описані в літературі стрибки — артефакт метрики. Це
доводить слабше й надійніше: **розривна метрика перетворює гладке покращення на
стрибок сама, без жодної допомоги з боку моделі**, — тож побачивши стрибок,
треба спершу порахувати ту саму річ гладкою лінійкою.

## 9 · Що ми заміряли, а що ні

Чесний перелік меж цього зошита.

* Наш діапазон розмірів — кілька разів за всіма вагами. У літературі криві
  масштабу будують на шести порядках. Ми відтворили **форму**, а не масштаб.
* Одне зерно на точку. Різниця, менша за розкид між зернами, не є різницею, а
  розкиду ми тут не міряли — це перше із завдань нижче.
* Крок навчання один на всі розміри, і найширшій моделі він, найімовірніше,
  завеликий. Заміряний показник — нижня оцінка.
* Точний збіг у нас дорівнює нулю в **усіх** точках. Це означає не «стрибка
  немає», а «ця метрика на цьому діапазоні не має роздільної здатності»: вона
  однаково оцінює найгіршу й найкращу з наших моделей.

In [ ]:
print(f'увесь зошит: {time.process_time() - started_at:.0f} с процесорного часу')

## Завдання

### 🟢 Рівень 1 — База

Додай до кривої за розміром **друге зерно** (`SEED = 1`) і перебудуй обидва
підгони.

**Зроблено, якщо:** для кожної ширини надруковано два значення нат, названо
розкид між зернами, і сказано, чи є різниця між сусідніми ширинами більшою за
цей розкид.

### 🟡 Рівень 2 — Плюс

Побудуй криву за обсягом даних для **трансформера**, а не для біграми: три
частки навчальної вибірки при сталій ширині. Порівняй показник із біграмним.

**Зроблено, якщо:** надруковано три точки, показник і `R²`, і словами сказано,
чий показник крутіший і що це означає для того, у що вигідніше вкладати —
у ваги чи в дані.

### 🔴 Рівень 3 — Виклик

Знайди, де саме зʼявився б «стрибок». Використай підгін `k=1` за розміром, щоб
передбачити, при якій кількості ваг збіг перших чотирьох токенів перетне 1 %.
Потім чесно назви, наскільки далеко це передбачення виходить за межі заміряного.

**Зроблено, якщо:** названо конкретну кількість ваг, у скільки разів вона більша
за найбільшу заміряну модель, і пояснено, чому це число є гіпотезою, а не
прогнозом.

## Підказки

* Розкид між зернами дешевше міряти на найменшій моделі: якщо він там уже
  більший за різницю між ширинами, більші моделі нічого не врятують.
* Частку даних беруть **випадковою вибіркою**, а не першими рядками: корпус
  лежить у порядку програм.
* Щоб перевернути підгін і дістати `x` із заданого `y`, потрібна та сама формула
  прямої, розвʼязана відносно `log x` — вона вже написана в `tokens_needed`.